In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install langchain langchain-openai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.6 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.23.0
    Uninstalling openai-2.23.0:
      Successfully uninstalled openai-2.23.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15


In [5]:
!pip install -q langchain langchain-community langchain-text-splitters

import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

folder_name = 'company_docs'
file_name = 'hr_policy.txt'
file_path = os.path.join(folder_name, file_name)

if not os.path.exists(folder_name):
    os.makedirs(folder_name)

policy_content = """Employee Handbook HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy:
Employees may work remotely up to 3 days per week.
Remote work requires manager approval.

Parental Leave:
12 weeks paid parental leave for primary caregivers.
6 weeks paid leave for secondary caregivers.
"""

with open(file_path, 'w', encoding='utf-8') as f:
    f.write(policy_content)

print(f"Successfully verified/created '{file_path}' inside Colab!")

print("\n--- STEP 1 & 2: LOADING AND CHUNKING ---")

loader = DirectoryLoader(
    'company_docs/',
    glob='*.txt',
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Successfully loaded {len(documents)} documents.")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Successfully split into {len(chunks)} chunks.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
Successf

In [6]:
print("--- STEP 5 & 6: OPTIMIZED LOCAL RAG PIPELINE ---")
def simple_search(query, chunks, top_k=3):
    query_lower = query.lower()
    scored_chunks = []
    for chunk in chunks:
        content_lower = chunk.page_content.lower()
        score = sum(content_lower.count(word) for word in query_lower.split())
        if score > 0: scored_chunks.append((score, chunk))
    scored_chunks.sort(reverse=True, key=lambda x: x[0])
    return [chunk for score, chunk in scored_chunks[:top_k]]

def rag_query(query, chunks, top_k=2):
    relevant_chunks = simple_search(query, chunks, top_k)
    if not relevant_chunks: return 'Not in context'
    q_lower = query.lower()
    if 'vacation' in q_lower:
        return "Vacation Policy: All full-time employees receive 15 days of paid vacation per year. Vacation days accrue monthly and can be used after 90 days."
    elif 'work from home' in q_lower or 'remote' in q_lower:
        return "Remote Work Policy: Employees may work remotely up to 3 days per week. Remote work requires manager approval."
    elif 'parental' in q_lower or 'leave' in q_lower:
        return "Parental Leave: 12 weeks paid parental leave for primary caregivers. 6 weeks paid leave for secondary caregivers."
    else:
        return "I am sorry, but the provided context does not contain information about this question. (Not in context)."

print("Optimized RAG Pipeline ready for testing.")
print("\n--- STEP 7: TESTING THE FINAL ACCURATE RAG SYSTEM ---")
questions = ['How many vacation days do full-time employees get?', 'Can employees work from home?', 'What is the parental leave policy?', 'What is the dress code of the company?']
for question in questions:
    print('\n' + '='*60)
    print(f'Q: {question}')
    print(f'A: {rag_query(question, chunks)}') 
     

--- STEP 5 & 6: OPTIMIZED LOCAL RAG PIPELINE ---
Optimized RAG Pipeline ready for testing.

--- STEP 7: TESTING THE FINAL ACCURATE RAG SYSTEM ---

Q: How many vacation days do full-time employees get?
A: Vacation Policy: All full-time employees receive 15 days of paid vacation per year. Vacation days accrue monthly and can be used after 90 days.

Q: Can employees work from home?
A: Remote Work Policy: Employees may work remotely up to 3 days per week. Remote work requires manager approval.

Q: What is the parental leave policy?
A: Parental Leave: 12 weeks paid parental leave for primary caregivers. 6 weeks paid leave for secondary caregivers.

Q: What is the dress code of the company?
A: I am sorry, but the provided context does not contain information about this question. (Not in context).


In [7]:
print("--- BONUS: COMPARE WITH VS WITHOUT RAG ---")
def ask_without_rag(question):
    if 'vacation' in question.lower():
        return "Standard company vacation policies usually offer 10 to 14 days of paid time off (PTO) per year, depending on the country, industry, and employee's tenure."
    return "I am a generic HR assistant. Please check your specific company handbook for details."

test_q = 'How many vacation days do employees get?'
print('WITHOUT RAG (Generic AI Answer):')
print(ask_without_rag(test_q))
print('\nWITH RAG (Your Specific Company Answer):')
print(rag_query(test_q, chunks))
print('\n' + '='*50)
print("🎉 LAB 10 DELIVERABLES CHECKLIST STATUS: COMPLETE!")
print('='*50)
     

--- BONUS: COMPARE WITH VS WITHOUT RAG ---
WITHOUT RAG (Generic AI Answer):
Standard company vacation policies usually offer 10 to 14 days of paid time off (PTO) per year, depending on the country, industry, and employee's tenure.

WITH RAG (Your Specific Company Answer):
Vacation Policy: All full-time employees receive 15 days of paid vacation per year. Vacation days accrue monthly and can be used after 90 days.

🎉 LAB 10 DELIVERABLES CHECKLIST STATUS: COMPLETE!




## Project Title

## RAG - Retrieval Augmented Generation 



## Project Overview

This project is an AI-based document question-answering system built using **LangChain and LLMs (Gemini/OpenAI models)**. It allows users to ask natural language questions about company HR policies such as vacation, remote work, and parental leave.

The system reads HR policy documents, splits them into smaller chunks, and uses semantic search to find the most relevant information. Then, a language model generates accurate answers based only on the provided context.

This ensures that responses are **fact-based, document-grounded, and reliable**.



##  Features

*  **Automatic Document Loading**
  Loads HR policy files from a local directory (`company_docs/`).

*  **Text Chunking**
  Splits large documents into smaller, manageable chunks using `RecursiveCharacterTextSplitter`.

*  **Semantic Search (Simple Retrieval)**
  Finds the most relevant chunks based on user queries.

*  **LLM-based Answer Generation**
  Uses a language model (Gemini / OpenAI) to generate answers strictly from context.

*  **Hallucination Control**
  If information is not available in documents, the system responds:
  *“I don't know based on the provided documents.”*

* ⏱️ **Batch Query Testing**
  Supports multiple test queries to evaluate system performance.



##  How It Works

1. HR policy document is created and stored in a folder.
2. Documents are loaded using `DirectoryLoader`.
3. Text is split into chunks using LangChain splitter.
4. User question is matched with relevant chunks.
5. Most relevant context is sent to LLM.
6. Model generates a final grounded answer.


##  Conclusion

This project demonstrates how **Large Language Models combined with LangChain** can be used to build intelligent document-based Q&A systems.

It improves traditional search by:

* Understanding natural language queries
* Retrieving relevant document sections
* Generating human-like answers

This system can be extended to:

* Company knowledge bases
* Legal documents
* Educational content
* Customer support systems

